# Import BLHS 2025 vào Neo4j + quan hệ EXCEPTION

Notebook này:
1. Kết nối Neo4j (driver Python).
2. Chạy các câu `LOAD CSV` để nạp `blhs_enriched.csv` (cần đặt file vào thư mục **import** của Neo4j).
3. Đọc `khong_thuoc_data.csv`, xác định nút nguồn (Điểm / Khoản / Điều) và tạo cạnh `[:EXCEPTION]` tới các nút `Dieu` được tham chiếu trong văn bản (trích số Điều bằng regex).

**Chuẩn bị Neo4j Desktop / Docker:** Sao chép `blhs_enriched.csv` và `khong_thuoc_data.csv` vào thư mục import (ví dụ `neo4j/import/`). Trong `neo4j.conf`, `dbms.directories.import` trỏ tới thư mục đó.

## 1. Cài đặt và cấu hình

In [10]:
%pip install -q neo4j pandas

Note: you may need to restart the kernel to use updated packages.


In [11]:
import re
from pathlib import Path

import pandas as pd
from neo4j import GraphDatabase

# --- Sửa cho phù hợp môi trường ---
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "12345678"

# Đường dẫn CSV trong project (dùng cho bước Python; LOAD CSV vẫn dùng file:/// trên server Neo4j)
BASE = Path(r"C:\Users\Admin\Desktop\DATN")
CSV_KHONG_THUOC = BASE / "khong_thuoc_data.csv"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))


def run_cypher(query: str, params: dict | None = None):
    """Thực thi một câu Cypher và trả về summary."""
    params = params or {}
    with driver.session() as session:
        result = session.run(query, params)
        summary = result.consume()
        counters = summary.counters
        return {
            "nodes_created": counters.nodes_created,
            "relationships_created": counters.relationships_created,
            "properties_set": counters.properties_set,
        }


driver.verify_connectivity()
print("Đã kết nối Neo4j.")

Đã kết nối Neo4j.


## 2. Import dữ liệu chính (`blhs_enriched.csv`)

Điều kiện: file `blhs_enriched.csv` nằm trong thư mục import của Neo4j để URI `file:///blhs_enriched.csv` hoạt động.

In [12]:
IMPORT_PHAN = """
LOAD CSV WITH HEADERS FROM 'file:///blhs_enriched.csv' AS row
WITH row
WHERE row.`Phần` IS NOT NULL AND trim(row.`Phần`) <> ''
MERGE (p:Phan {ten: trim(row.`Phần`)});
"""

IMPORT_CHUONG = """
LOAD CSV WITH HEADERS FROM 'file:///blhs_enriched.csv' AS row
WITH row
WHERE row.`Phần` IS NOT NULL AND trim(row.`Phần`) <> ''
  AND row.`Chương` IS NOT NULL AND trim(row.`Chương`) <> ''
MERGE (p:Phan {ten: trim(row.`Phần`)})
MERGE (c:Chuong {id: trim(row.`Phần`) + '|' + trim(row.`Chương`)})
SET c.ten = trim(row.`Chương`)
MERGE (p)-[:HAS_CHUONG]->(c);
"""

IMPORT_DIEU = """
LOAD CSV WITH HEADERS FROM 'file:///blhs_enriched.csv' AS row
WITH row
WHERE row.`Phần` IS NOT NULL AND trim(row.`Phần`) <> ''
  AND row.`Chương` IS NOT NULL AND trim(row.`Chương`) <> ''
  AND row.`Số điều` IS NOT NULL AND trim(row.`Số điều`) <> ''
WITH row,
     trim(row.`Phần`) AS phan,
     trim(row.`Chương`) AS chuong,
     trim(row.`Số điều`) AS so_dieu_raw,
     toFloat(trim(row.`Số điều`)) AS so_dieu_float,
     CASE
        WHEN trim(row.`Số điều`) ENDS WITH '.0'
        THEN left(trim(row.`Số điều`), size(trim(row.`Số điều`)) - 2)
        ELSE trim(row.`Số điều`)
     END AS so_dieu_norm
MERGE (c:Chuong {id: phan + '|' + chuong})
MERGE (d:Dieu {id: phan + '|' + chuong + '|Điều ' + so_dieu_norm})
SET d.so_dieu = so_dieu_float,
    d.tieu_de = trim(coalesce(row.`Tiêu đề điều`, '')),
    d.noi_dung = trim(coalesce(row.`Nội dung điều`, '')),
    d.full_text = trim(coalesce(row.`full_text`, '')),
    d.crime_group = trim(coalesce(row.`crime_group`, '')),
    d.crime_category = trim(coalesce(row.`crime_category`, '')),
    d.intent_type = trim(coalesce(row.`intent_type`, '')),
    d.actor = trim(coalesce(row.`actor`, '')),
    d.victim = trim(coalesce(row.`victim`, '')),
    d.action = trim(coalesce(row.`action`, '')),
    d.consequence = trim(coalesce(row.`consequence`, '')),
    d.penalty_main = trim(coalesce(row.`penalty_main`, '')),
    d.penalty_supplementary = trim(coalesce(row.`penalty_supplementary`, '')),
    d.aggravating_circ = trim(coalesce(row.`aggravating_circ`, '')),
    d.mitigating_circ = trim(coalesce(row.`mitigating_circ`, '')),
    d.condition = trim(coalesce(row.`condition`, '')),
    d.exception = trim(coalesce(row.`exception`, '')),
    d.judicial_measure = trim(coalesce(row.`judicial_measure`, ''))
MERGE (c)-[:HAS_DIEU]->(d);
"""

IMPORT_KHOAN = """
LOAD CSV WITH HEADERS FROM 'file:///blhs_enriched.csv' AS row
WITH row
WHERE row.`Phần` IS NOT NULL AND trim(row.`Phần`) <> ''
  AND row.`Chương` IS NOT NULL AND trim(row.`Chương`) <> ''
  AND row.`Số điều` IS NOT NULL AND trim(row.`Số điều`) <> ''
  AND row.`Số khoản` IS NOT NULL AND trim(row.`Số khoản`) <> ''
WITH row,
     trim(row.`Phần`) AS phan,
     trim(row.`Chương`) AS chuong,
     trim(row.`Số điều`) AS so_dieu_raw,
     CASE
        WHEN trim(row.`Số điều`) ENDS WITH '.0'
        THEN left(trim(row.`Số điều`), size(trim(row.`Số điều`)) - 2)
        ELSE trim(row.`Số điều`)
     END AS so_dieu_norm,
     trim(row.`Số khoản`) AS so_khoan_raw,
     CASE
        WHEN trim(row.`Số khoản`) ENDS WITH '.0'
        THEN left(trim(row.`Số khoản`), size(trim(row.`Số khoản`)) - 2)
        ELSE trim(row.`Số khoản`)
     END AS so_khoan_norm,
     toInteger(toFloat(row.`Số khoản`)) AS so_khoan_int
MERGE (d:Dieu {id: phan + '|' + chuong + '|Điều ' + so_dieu_norm})
MERGE (k:Khoan {id: phan + '|' + chuong + '|Điều ' + so_dieu_norm + '|Khoản ' + so_khoan_norm})
SET k.so_khoan = so_khoan_int,
    k.noi_dung = trim(coalesce(row.`Nội dung khoản`, '')),
    k.full_text = trim(coalesce(row.`full_text`, '')),
    k.crime_group = trim(coalesce(row.`crime_group`, '')),
    k.crime_category = trim(coalesce(row.`crime_category`, '')),
    k.intent_type = trim(coalesce(row.`intent_type`, '')),
    k.actor = trim(coalesce(row.`actor`, '')),
    k.victim = trim(coalesce(row.`victim`, '')),
    k.action = trim(coalesce(row.`action`, '')),
    k.consequence = trim(coalesce(row.`consequence`, '')),
    k.penalty_main = trim(coalesce(row.`penalty_main`, '')),
    k.penalty_supplementary = trim(coalesce(row.`penalty_supplementary`, '')),
    k.aggravating_circ = trim(coalesce(row.`aggravating_circ`, '')),
    k.mitigating_circ = trim(coalesce(row.`mitigating_circ`, '')),
    k.condition = trim(coalesce(row.`condition`, '')),
    k.exception = trim(coalesce(row.`exception`, '')),
    k.judicial_measure = trim(coalesce(row.`judicial_measure`, ''))
MERGE (d)-[:HAS_KHOAN]->(k);
"""

IMPORT_DIEM = """
LOAD CSV WITH HEADERS FROM 'file:///blhs_enriched.csv' AS row
WITH row
WHERE row.`Phần` IS NOT NULL AND trim(row.`Phần`) <> ''
  AND row.`Chương` IS NOT NULL AND trim(row.`Chương`) <> ''
  AND row.`Số điều` IS NOT NULL AND trim(row.`Số điều`) <> ''
  AND row.`Số khoản` IS NOT NULL AND trim(row.`Số khoản`) <> ''
  AND row.`Điểm` IS NOT NULL AND trim(row.`Điểm`) <> ''
WITH row,
     trim(row.`Phần`) AS phan,
     trim(row.`Chương`) AS chuong,
     trim(row.`Số điều`) AS so_dieu_raw,
     CASE
        WHEN trim(row.`Số điều`) ENDS WITH '.0'
        THEN left(trim(row.`Số điều`), size(trim(row.`Số điều`)) - 2)
        ELSE trim(row.`Số điều`)
     END AS so_dieu_norm,
     trim(row.`Số khoản`) AS so_khoan_raw,
     CASE
        WHEN trim(row.`Số khoản`) ENDS WITH '.0'
        THEN left(trim(row.`Số khoản`), size(trim(row.`Số khoản`)) - 2)
        ELSE trim(row.`Số khoản`)
     END AS so_khoan_norm,
     trim(row.`Điểm`) AS diem_raw
MERGE (k:Khoan {id: phan + '|' + chuong + '|Điều ' + so_dieu_norm + '|Khoản ' + so_khoan_norm})
MERGE (di:Diem {id: phan + '|' + chuong + '|Điều ' + so_dieu_norm + '|Khoản ' + so_khoan_norm + '|Điểm ' + diem_raw})
SET di.ky_hieu = diem_raw,
    di.noi_dung = trim(coalesce(row.`Nội dung điểm`, '')),
    di.full_text = trim(coalesce(row.`full_text`, '')),
    di.crime_group = trim(coalesce(row.`crime_group`, '')),
    di.crime_category = trim(coalesce(row.`crime_category`, '')),
    di.intent_type = trim(coalesce(row.`intent_type`, '')),
    di.actor = trim(coalesce(row.`actor`, '')),
    di.victim = trim(coalesce(row.`victim`, '')),
    di.action = trim(coalesce(row.`action`, '')),
    di.consequence = trim(coalesce(row.`consequence`, '')),
    di.penalty_main = trim(coalesce(row.`penalty_main`, '')),
    di.penalty_supplementary = trim(coalesce(row.`penalty_supplementary`, '')),
    di.aggravating_circ = trim(coalesce(row.`aggravating_circ`, '')),
    di.mitigating_circ = trim(coalesce(row.`mitigating_circ`, '')),
    di.condition = trim(coalesce(row.`condition`, '')),
    di.exception = trim(coalesce(row.`exception`, '')),
    di.judicial_measure = trim(coalesce(row.`judicial_measure`, ''))
MERGE (k)-[:HAS_DIEM]->(di);
"""

for name, q in [
    ("Phần", IMPORT_PHAN),
    ("Chương", IMPORT_CHUONG),
    ("Điều", IMPORT_DIEU),
    ("Khoản", IMPORT_KHOAN),
    ("Điểm", IMPORT_DIEM),
]:
    stats = run_cypher(q)
    print(name, stats)

Phần {'nodes_created': 3, 'relationships_created': 0, 'properties_set': 3}
Chương {'nodes_created': 27, 'relationships_created': 27, 'properties_set': 3667}
Điều {'nodes_created': 427, 'relationships_created': 427, 'properties_set': 65947}
Khoản {'nodes_created': 1369, 'relationships_created': 1369, 'properties_set': 62484}
Điểm {'nodes_created': 2873, 'relationships_created': 2873, 'properties_set': 51833}


> **Ghi chú:** `d.so_dieu` dùng `toFloat` để giữ đúng số điều dạng `217.5` (tránh `toInteger` làm tròn). Câu gốc của bạn dùng `toInteger(row.\`Số điều\`)` — có thể lỗi với điều có phần thập phân.

Nếu đã import trước đó với `toInteger`, có thể sửa lại:
```cypher
MATCH (d:Dieu)
SET d.so_dieu = toFloat(split(d.id, 'Điều ')[-1]);
```

## 3. Quan hệ `EXCEPTION` từ `khong_thuoc_data.csv`

- **Nguồn:** nút `Diem`, `Khoan` hoặc `Dieu` (ưu tiên chi tiết nhất có trong dòng: có Điểm → `Diem`, không thì có Khoản → `Khoan`, còn lại → `Dieu`).
- **Đích:** các nút `Dieu` có `id` kết thúc bằng `|Điều {số}` khớp số được trích từ văn bản (mẫu `Điều 66`, `các điều 134, 141`, `khoản 2 Điều 66`).
- **Loại trừ:** không nối tới chính số điều của dòng hiện tại (tránh tự tham chiếu khi có cụm "Điều này").

Có thể đổi tên quan hệ thành `EXEPTION` nếu bạn cần khớp tên gọi nội bộ — thay `EXCEPTION` trong biến `REL_TYPE` bên dưới.

In [13]:
REL_TYPE = "EXCEPTION"  # Đổi thành "EXEPTION" nếu bạn muốn đúng chính tả cũ

# Trích số điều từ văn bản (chuỗi số có thể là 217.5)
_RE_DIEU = re.compile(r"Điều\s+(\d+(?:\.\d+)?)", re.IGNORECASE)
_RE_CAC_DIEU = re.compile(r"các\s+điều\s+([0-9,\s]+(?:và\s+\d+)?)", re.IGNORECASE)
_RE_KHOAN_DIEU = re.compile(r"khoản\s+\d+\s+Điều\s+(\d+(?:\.\d+)?)", re.IGNORECASE)


def _norm_so_dieu(s: str) -> str:
    return str(s).strip().replace(",", ".")


def _only_exception_clauses(text: str) -> str:
    """Chỉ giữ câu/cụm có 'không thuộc' hoặc 'trừ tội phạm' để tránh nối nhầm danh sách điều dài (vd. Điều 389)."""
    if not text or not str(text).strip():
        return ""
    parts = re.split(r"(?<=[.;])\s+", text)
    kept = []
    for p in parts:
        pl = p.lower()
        if "không thuộc" in pl or "trừ tội phạm" in pl:
            kept.append(p)
    if kept:
        return " ".join(kept)
    # Fallback: cả đoạn là một câu dài không bị tách bởi . hoặc ;
    if re.search(r"không thuộc|trừ\s+tội\s+phạm", text, re.I):
        return text
    return ""


def extract_target_dieu_numbers(text: str, current_so_dieu: str) -> set[str]:
    """Trích số điều tham chiếu từ các cụm ngoại lệ; bỏ trùng số điều hiện tại."""
    scoped = _only_exception_clauses(text)
    if not scoped:
        return set()
    cur = _norm_so_dieu(current_so_dieu)
    found: set[str] = set()
    for m in _RE_DIEU.finditer(scoped):
        num = m.group(1)
        if num != cur:
            found.add(num)
    for m in _RE_KHOAN_DIEU.finditer(scoped):
        found.add(m.group(1))
    for m in _RE_CAC_DIEU.finditer(scoped):
        chunk = m.group(1)
        for part in re.split(r"[;,\s]+và\s+|\s+và\s+|,|;", chunk):
            part = part.strip()
            if not part:
                continue
            if part.isdigit() or re.match(r"^\d+\.\d+$", part):
                if part != cur:
                    found.add(part)
    found.discard(cur)
    return found


def build_source_id(row: pd.Series) -> tuple[str, str]:
    """Trả về (label, id): Diem | Khoan | Dieu"""
    phan = str(row["Phần"]).strip()
    chuong = str(row["Chương"]).strip()
    so_dieu = str(row["Số điều"]).strip()
    base = f"{phan}|{chuong}|Điều {so_dieu}"
    sk = row.get("Số khoản")
    diem = row.get("Điểm")
    has_khoan = pd.notna(sk) and str(sk).strip() != ""
    has_diem = pd.notna(diem) and str(diem).strip() != ""
    if has_khoan and has_diem:
        return "Diem", f"{base}|Khoản {str(sk).strip()}|Điểm {str(diem).strip()}"
    if has_khoan:
        return "Khoan", f"{base}|Khoản {str(sk).strip()}"
    return "Dieu", base


def combined_text(row: pd.Series) -> str:
    parts = []
    for col in ["Nội dung điều", "Nội dung khoản", "Nội dung điểm"]:
        v = row.get(col)
        if pd.notna(v) and str(v).strip():
            parts.append(str(v))
    return " ".join(parts)


MERGE_EXCEPTION = f"""
UNWIND $rows AS r
MATCH (s)
WHERE (s:Diem OR s:Khoan OR s:Dieu) AND s.id = r.source_id
WITH s, r.targets AS targets
UNWIND targets AS tnum
MATCH (td:Dieu)
WHERE td.id ENDS WITH ('|Điều ' + tnum)
MERGE (s)-[:{REL_TYPE}]->(td)
"""


def create_exception_edges(batch_size: int = 200):
    df = pd.read_csv(CSV_KHONG_THUOC, dtype=str).fillna("")
    batches = []
    current = []
    skipped = 0
    for _, row in df.iterrows():
        label, sid = build_source_id(row)
        text = combined_text(row)
        so_dieu = str(row["Số điều"]).strip()
        targets = sorted(extract_target_dieu_numbers(text, so_dieu))
        if not targets:
            skipped += 1
            continue
        current.append({"source_id": sid, "targets": targets})
        if len(current) >= batch_size:
            batches.append(current)
            current = []
    if current:
        batches.append(current)

    total_rels = 0
    for batch in batches:
        with driver.session() as session:
            result = session.run(MERGE_EXCEPTION, {"rows": batch})
            summary = result.consume()
            total_rels += summary.counters.relationships_created
    print(f"Đã xử lý {len(df)} dòng; không có mục tiêu trích được: {skipped}; cạnh {REL_TYPE} tạo (ước lượng batch): {total_rels}")


create_exception_edges()

Đã xử lý 143 dòng; không có mục tiêu trích được: 8; cạnh EXCEPTION tạo (ước lượng batch): 561


## 4. Kiểm tra nhanh

Ví dụ: đếm một phần quan hệ `EXCEPTION` và xem một cạnh mẫu.

In [14]:
CHECK = f"""
MATCH ()-[r:{REL_TYPE}]->()
RETURN count(r) AS total;
"""
with driver.session() as s:
    print(s.run(CHECK).single())

SAMPLE = f"""
MATCH (s)-[r:{REL_TYPE}]->(t:Dieu)
RETURN labels(s)[0] AS loai_nguon, s.id AS nguon, t.id AS dich
LIMIT 5;
"""
with driver.session() as s:
    for rec in s.run(SAMPLE):
        print(dict(rec))

<Record total=561>
{'loai_nguon': 'Diem', 'nguon': 'Phần thứ nhất - NHỮNG QUY ĐỊNH CHUNG|Chương XII - NHỮNG QUY ĐỊNH ĐỐI VỚI NGƯỜI DƯỚI 18 TUỔI PHẠM TỘI|Điều 91|Khoản 2|Điểm b', 'dich': 'Phần thứ nhất - NHỮNG QUY ĐỊNH CHUNG|Chương III - TỘI PHẠM|Điều 12'}
{'loai_nguon': 'Khoan', 'nguon': 'Phần thứ hai - CÁC TỘI PHẠM|Chương XXIV - CÁC TỘI XÂM PHẠM HOẠT ĐỘNG TƯ PHÁP|Điều 390|Khoản 1', 'dich': 'Phần thứ nhất - NHỮNG QUY ĐỊNH CHUNG|Chương III - TỘI PHẠM|Điều 14'}
{'loai_nguon': 'Diem', 'nguon': 'Phần thứ hai - CÁC TỘI PHẠM|Chương XXIV - CÁC TỘI XÂM PHẠM HOẠT ĐỘNG TƯ PHÁP|Điều 389|Khoản 1|Điểm a', 'dich': 'Phần thứ nhất - NHỮNG QUY ĐỊNH CHUNG|Chương III - TỘI PHẠM|Điều 18'}
{'loai_nguon': 'Diem', 'nguon': 'Phần thứ hai - CÁC TỘI PHẠM|Chương XXIV - CÁC TỘI XÂM PHẠM HOẠT ĐỘNG TƯ PHÁP|Điều 389|Khoản 1|Điểm b', 'dich': 'Phần thứ nhất - NHỮNG QUY ĐỊNH CHUNG|Chương III - TỘI PHẠM|Điều 18'}
{'loai_nguon': 'Diem', 'nguon': 'Phần thứ hai - CÁC TỘI PHẠM|Chương XXIV - CÁC TỘI XÂM PHẠM HOẠT ĐỘNG TƯ PHÁ

## 5. (Tuỳ chọn) `LOAD CSV` thuần cho `khong_thuoc` — chỉ nối theo **cùng một cấu trúc id**

Nếu sau này bạn xuất thêm cột `so_dieu_dich` trong CSV, có thể dùng mẫu sau trên Neo4j (không cần Python):

```cypher
LOAD CSV WITH HEADERS FROM 'file:///khong_thuoc_data.csv' AS row
WITH row
WHERE row.`Phần` IS NOT NULL AND trim(row.`Phần`) <> ''
  AND row.`Chương` IS NOT NULL AND trim(row.`Chương`) <> ''
  AND row.`Số điều` IS NOT NULL AND trim(row.`Số điều`) <> ''
  AND row.`so_dieu_dich` IS NOT NULL AND trim(row.`so_dieu_dich`) <> ''
WITH trim(row.`Phần`) AS phan, trim(row.`Chương`) AS chuong,
     trim(row.`Số điều`) AS so_dieu,
     trim(row.`Số khoản`) AS so_khoan,
     trim(row.`Điểm`) AS diem,
     trim(row.`so_dieu_dich`) AS dich
WITH phan, chuong, so_dieu, so_khoan, diem, dich,
     phan + '|' + chuong + '|Điều ' + so_dieu AS base
WITH dich,
  CASE WHEN diem IS NOT NULL AND diem <> '' THEN base + '|Khoản ' + so_khoan + '|Điểm ' + diem
       WHEN so_khoan IS NOT NULL AND so_khoan <> '' THEN base + '|Khoản ' + so_khoan
       ELSE base END AS source_id
MATCH (s {{id: source_id}})
MATCH (t:Dieu) WHERE t.id ENDS WITH '|Điều ' + dich
MERGE (s)-[:EXCEPTION]->(t);
```

Hiện tại file `khong_thuoc_data.csv` không có cột đích tường minh — notebook dùng Python + regex như trên.

## 6. Xây semantic graph từ thuộc tính BLHS

Ở trên, các thuộc tính như `crime_group`, `crime_category`, `intent_type`, `actor`, `victim`, `action`, `consequence`, `penalty_main`, `penalty_supplementary`, `aggravating_circ`, `mitigating_circ`, `condition`, `judicial_measure` đang được lưu **dạng chuỗi** trên các nút `Dieu`, `Khoan`, `Diem`.

Để LLM suy luận tội danh, ta chuyển các trường này thành **entity riêng** (node) và quan hệ, cho phép query dạng:

- Điều/khoản/điểm → nhóm tội, loại tội, ý thức chủ quan (cố ý/vô ý…), chủ thể, khách thể, hành vi, hậu quả, hình phạt chính/bổ sung, tình tiết tăng nặng/giảm nhẹ, điều kiện áp dụng, biện pháp tư pháp.
- Một tình tiết hay hình phạt có thể liên kết tới **nhiều** điều/khoản/điểm, giúp match nhiều tội danh trong cùng một tình huống.

In [15]:
# 6.1. Chuẩn hóa & tách multi-value, tạo entity semantic

SEMANTIC_QUERIES = [
    # crime_group
    """
    MATCH (n)
    WHERE (n:Dieu OR n:Khoan OR n:Diem) AND n.crime_group IS NOT NULL AND trim(n.crime_group) <> ''
    WITH n, [x IN split(n.crime_group, ';') | trim(x)] AS values
    UNWIND values AS v
    WITH n, v WHERE v <> ''
    MERGE (g:CrimeGroup {name: v})
    MERGE (n)-[:HAS_CRIME_GROUP]->(g);
    """,
    # crime_category
    """
    MATCH (n)
    WHERE (n:Dieu OR n:Khoan OR n:Diem) AND n.crime_category IS NOT NULL AND trim(n.crime_category) <> ''
    WITH n, [x IN split(n.crime_category, ';') | trim(x)] AS values
    UNWIND values AS v
    WITH n, v WHERE v <> ''
    MERGE (c:CrimeCategory {name: v})
    MERGE (n)-[:HAS_CRIME_CATEGORY]->(c);
    """,
    # intent_type
    """
    MATCH (n)
    WHERE (n:Dieu OR n:Khoan OR n:Diem) AND n.intent_type IS NOT NULL AND trim(n.intent_type) <> ''
    WITH n, [x IN split(n.intent_type, ';') | trim(x)] AS values
    UNWIND values AS v
    WITH n, v WHERE v <> ''
    MERGE (i:IntentType {name: v})
    MERGE (n)-[:HAS_INTENT]->(i);
    """,
    # actor
    """
    MATCH (n)
    WHERE (n:Dieu OR n:Khoan OR n:Diem) AND n.actor IS NOT NULL AND trim(n.actor) <> ''
    WITH n, [x IN split(n.actor, ';') | trim(x)] AS values
    UNWIND values AS v
    WITH n, v WHERE v <> ''
    MERGE (a:ActorRole {name: v})
    MERGE (n)-[:HAS_ACTOR]->(a);
    """,
    # victim
    """
    MATCH (n)
    WHERE (n:Dieu OR n:Khoan OR n:Diem) AND n.victim IS NOT NULL AND trim(n.victim) <> ''
    WITH n, [x IN split(n.victim, ';') | trim(x)] AS values
    UNWIND values AS v
    WITH n, v WHERE v <> ''
    MERGE (vct:VictimType {name: v})
    MERGE (n)-[:HAS_VICTIM]->(vct);
    """,
    # action
    """
    MATCH (n)
    WHERE (n:Dieu OR n:Khoan OR n:Diem) AND n.action IS NOT NULL AND trim(n.action) <> ''
    WITH n, [x IN split(n.action, ';') | trim(x)] AS values
    UNWIND values AS v
    WITH n, v WHERE v <> ''
    MERGE (ac:ActionType {name: v})
    MERGE (n)-[:HAS_ACTION]->(ac);
    """,
    # consequence
    """
    MATCH (n)
    WHERE (n:Dieu OR n:Khoan OR n:Diem) AND n.consequence IS NOT NULL AND trim(n.consequence) <> ''
    WITH n, [x IN split(n.consequence, ';') | trim(x)] AS values
    UNWIND values AS v
    WITH n, v WHERE v <> ''
    MERGE (cs:ConsequenceType {name: v})
    MERGE (n)-[:HAS_CONSEQUENCE]->(cs);
    """,
    # penalty_main
    """
    MATCH (n)
    WHERE (n:Dieu OR n:Khoan OR n:Diem) AND n.penalty_main IS NOT NULL AND trim(n.penalty_main) <> ''
    WITH n, [x IN split(n.penalty_main, ';') | trim(x)] AS values
    UNWIND values AS v
    WITH n, v WHERE v <> ''
    MERGE (pm:PenaltyMain {name: v})
    MERGE (n)-[:HAS_PENALTY_MAIN]->(pm);
    """,
    # penalty_supplementary
    """
    MATCH (n)
    WHERE (n:Dieu OR n:Khoan OR n:Diem) AND n.penalty_supplementary IS NOT NULL AND trim(n.penalty_supplementary) <> ''
    WITH n, [x IN split(n.penalty_supplementary, ';') | trim(x)] AS values
    UNWIND values AS v
    WITH n, v WHERE v <> ''
    MERGE (ps:PenaltySupplementary {name: v})
    MERGE (n)-[:HAS_PENALTY_SUPP]->(ps);
    """,
    # aggravating circumstances
    """
    MATCH (n)
    WHERE (n:Dieu OR n:Khoan OR n:Diem) AND n.aggravating_circ IS NOT NULL AND trim(n.aggravating_circ) <> ''
    WITH n, [x IN split(n.aggravating_circ, ';') | trim(x)] AS values
    UNWIND values AS v
    WITH n, v WHERE v <> ''
    MERGE (ag:AggravatingCircumstance {name: v})
    MERGE (n)-[:HAS_AGGRAVATING]->(ag);
    """,
    # mitigating circumstances
    """
    MATCH (n)
    WHERE (n:Dieu OR n:Khoan OR n:Diem) AND n.mitigating_circ IS NOT NULL AND trim(n.mitigating_circ) <> ''
    WITH n, [x IN split(n.mitigating_circ, ';') | trim(x)] AS values
    UNWIND values AS v
    WITH n, v WHERE v <> ''
    MERGE (mt:MitigatingCircumstance {name: v})
    MERGE (n)-[:HAS_MITIGATING]->(mt);
    """,
    # condition
    """
    MATCH (n)
    WHERE (n:Dieu OR n:Khoan OR n:Diem) AND n.condition IS NOT NULL AND trim(n.condition) <> ''
    WITH n, [x IN split(n.condition, ';') | trim(x)] AS values
    UNWIND values AS v
    WITH n, v WHERE v <> ''
    MERGE (cd:Condition {name: v})
    MERGE (n)-[:HAS_CONDITION]->(cd);
    """,
    # judicial_measure
    """
    MATCH (n)
    WHERE (n:Dieu OR n:Khoan OR n:Diem) AND n.judicial_measure IS NOT NULL AND trim(n.judicial_measure) <> ''
    WITH n, [x IN split(n.judicial_measure, ';') | trim(x)] AS values
    UNWIND values AS v
    WITH n, v WHERE v <> ''
    MERGE (jm:JudicialMeasure {name: v})
    MERGE (n)-[:HAS_JUDICIAL_MEASURE]->(jm);
    """,
]

for q in SEMANTIC_QUERIES:
    stats = run_cypher(q)
    print(stats)


{'nodes_created': 27, 'relationships_created': 4669, 'properties_set': 27}
{'nodes_created': 5, 'relationships_created': 4669, 'properties_set': 5}
{'nodes_created': 4, 'relationships_created': 4669, 'properties_set': 4}
{'nodes_created': 3, 'relationships_created': 4669, 'properties_set': 3}
{'nodes_created': 9, 'relationships_created': 4708, 'properties_set': 9}
{'nodes_created': 26, 'relationships_created': 4742, 'properties_set': 26}
{'nodes_created': 9, 'relationships_created': 4821, 'properties_set': 9}
{'nodes_created': 51, 'relationships_created': 6175, 'properties_set': 51}
{'nodes_created': 6, 'relationships_created': 871, 'properties_set': 6}
{'nodes_created': 12, 'relationships_created': 636, 'properties_set': 12}
{'nodes_created': 20, 'relationships_created': 41, 'properties_set': 20}
{'nodes_created': 181, 'relationships_created': 312, 'properties_set': 181}
{'nodes_created': 5, 'relationships_created': 32, 'properties_set': 5}


## 7. Nâng cấp graph ngoại lệ `khong_thuoc_data.csv` thành node/edge semantic

Để tránh phải re-regex nhiều lần khi reasoning, ta tạo thêm lớp `ExceptionClause`:

- Mỗi dòng trong `khong_thuoc_data.csv` → một node `:ExceptionClause` chứa `text`, `source_id`, `so_dieu`, `phan`, `chuong`.
- Nguồn (`Diem`/`Khoan`/`Dieu`) liên kết tới exception bằng `[:HAS_EXCEPTION]`.
- Exception lại liên kết tới các `Dieu` bị loại trừ bằng `[:EXCEPTS_DIEU]`.

Như vậy LLM có thể query hoặc nhúng vector trên node `ExceptionClause` thay vì phải xử lý regex trên chuỗi thô mỗi lần.

In [16]:
# 7.1. Tạo node ExceptionClause + quan hệ semantic

MERGE_EXCEPTION_CLAUSE = """
UNWIND $rows AS r
MATCH (s)
WHERE (s:Diem OR s:Khoan OR s:Dieu) AND s.id = r.source_id
MERGE (e:ExceptionClause {id: r.source_id + '|' + r.row_index})
SET e.text = r.text,
    e.so_dieu = r.so_dieu,
    e.phan = r.phan,
    e.chuong = r.chuong
MERGE (s)-[:HAS_EXCEPTION]->(e)
WITH e, r.targets AS targets
UNWIND targets AS tnum
MATCH (td:Dieu)
WHERE td.id ENDS WITH ('|Điều ' + tnum)
MERGE (e)-[:EXCEPTS_DIEU]->(td);
"""


def create_exception_graph(batch_size: int = 200):
    df = pd.read_csv(CSV_KHONG_THUOC, dtype=str).fillna("")
    batches: list[list[dict]] = []
    current: list[dict] = []
    skipped = 0
    for idx, row in df.iterrows():
        label, sid = build_source_id(row)
        text = combined_text(row)
        so_dieu = str(row["Số điều"]).strip()
        targets = sorted(extract_target_dieu_numbers(text, so_dieu))
        if not targets:
            skipped += 1
            continue
        payload = {
            "row_index": str(idx),
            "source_id": sid,
            "so_dieu": so_dieu,
            "phan": str(row["Phần"]).strip(),
            "chuong": str(row["Chương"]).strip(),
            "text": text,
            "targets": targets,
        }
        current.append(payload)
        if len(current) >= batch_size:
            batches.append(current)
            current = []
    if current:
        batches.append(current)

    total_excepts = 0
    total_nodes = 0
    for batch in batches:
        with driver.session() as session:
            result = session.run(MERGE_EXCEPTION_CLAUSE, {"rows": batch})
            summary = result.consume()
            total_excepts += summary.counters.relationships_created
            total_nodes += summary.counters.nodes_created
    print(
        f"Đã xử lý {len(df)} dòng; bỏ qua {skipped}; "
        f"node ExceptionClause mới: {total_nodes}, quan hệ tạo thêm: {total_excepts}"
    )


# Gọi hàm sau khi đã có cấu trúc BLHS và (tuỳ chọn) quan hệ EXCEPTION cũ
create_exception_graph()

Đã xử lý 143 dòng; bỏ qua 8; node ExceptionClause mới: 135, quan hệ tạo thêm: 696


## 8. Normalize + index cho reasoning graph

Sau khi import xong semantic graph và `ExceptionClause`, ta chuẩn hóa thêm các trường text thành dạng không dấu để notebook reasoning so khớp ổn định hơn.

Đồng thời tạo index/constraint cho các node reasoning và các thuộc tính `*_norm` để bước truy vết về sau nhanh hơn.

In [17]:
import unicodedata


def normalize_text(text: str | None) -> str:
    if text is None or pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = text.replace("đ", "d").replace("Đ", "D")
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text


def fetch_all(query: str, params: dict | None = None):
    params = params or {}
    with driver.session() as session:
        return [dict(record) for record in session.run(query, params)]


def materialize_normalized_properties():
    label_props = [
        ("ActionType", "name", "name_norm"),
        ("VictimType", "name", "name_norm"),
        ("ConsequenceType", "name", "name_norm"),
        ("ActorRole", "name", "name_norm"),
        ("IntentType", "name", "name_norm"),
        ("CrimeCategory", "name", "name_norm"),
        ("CrimeGroup", "name", "name_norm"),
        ("PenaltyMain", "name", "name_norm"),
        ("ExceptionClause", "text", "text_norm"),
        ("Dieu", "tieu_de", "tieu_de_norm"),
        ("Dieu", "noi_dung", "noi_dung_norm"),
        ("Dieu", "full_text", "full_text_norm"),
        ("Khoan", "noi_dung", "noi_dung_norm"),
        ("Khoan", "full_text", "full_text_norm"),
        ("Diem", "noi_dung", "noi_dung_norm"),
        ("Diem", "full_text", "full_text_norm"),
    ]

    total = 0
    for label, source_prop, target_prop in label_props:
        rows = fetch_all(
            f"MATCH (n:{label}) WHERE n.{source_prop} IS NOT NULL RETURN elementId(n) AS eid, n.{source_prop} AS value"
        )
        for row in rows:
            run_cypher(
                f"MATCH (n) WHERE elementId(n) = $eid SET n.{target_prop} = $norm",
                {"eid": row["eid"], "norm": normalize_text(row["value"])},
            )
            total += 1
    return total


INDEX_AND_CONSTRAINT_QUERIES = [
    "CREATE CONSTRAINT reasoning_case_id IF NOT EXISTS FOR (n:ReasoningCase) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT reasoning_term_id IF NOT EXISTS FOR (n:ReasoningTerm) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT reasoning_candidate_id IF NOT EXISTS FOR (n:ReasoningCandidate) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT reasoning_step_id IF NOT EXISTS FOR (n:ReasoningStep) REQUIRE n.id IS UNIQUE",
    "CREATE INDEX action_type_name_norm IF NOT EXISTS FOR (n:ActionType) ON (n.name_norm)",
    "CREATE INDEX victim_type_name_norm IF NOT EXISTS FOR (n:VictimType) ON (n.name_norm)",
    "CREATE INDEX consequence_type_name_norm IF NOT EXISTS FOR (n:ConsequenceType) ON (n.name_norm)",
    "CREATE INDEX actor_role_name_norm IF NOT EXISTS FOR (n:ActorRole) ON (n.name_norm)",
    "CREATE INDEX intent_type_name_norm IF NOT EXISTS FOR (n:IntentType) ON (n.name_norm)",
    "CREATE INDEX crime_category_name_norm IF NOT EXISTS FOR (n:CrimeCategory) ON (n.name_norm)",
    "CREATE INDEX crime_group_name_norm IF NOT EXISTS FOR (n:CrimeGroup) ON (n.name_norm)",
    "CREATE INDEX penalty_main_name_norm IF NOT EXISTS FOR (n:PenaltyMain) ON (n.name_norm)",
    "CREATE INDEX exception_clause_text_norm IF NOT EXISTS FOR (n:ExceptionClause) ON (n.text_norm)",
    "CREATE INDEX dieu_full_text_norm IF NOT EXISTS FOR (n:Dieu) ON (n.full_text_norm)",
    "CREATE INDEX khoan_full_text_norm IF NOT EXISTS FOR (n:Khoan) ON (n.full_text_norm)",
    "CREATE INDEX diem_full_text_norm IF NOT EXISTS FOR (n:Diem) ON (n.full_text_norm)",
]

normalized_count = materialize_normalized_properties()
for query in INDEX_AND_CONSTRAINT_QUERIES:
    run_cypher(query)

print(f"Da bo sung normalize cho {normalized_count} node/field va tao xong index/constraint reasoning.")

Da bo sung normalize cho 10034 node/field va tao xong index/constraint reasoning.


In [18]:
driver.close()